# normal/anti集計の再確認

このNotebookは同じフォルダの公開用JSON/SQLiteだけを読みます。API・モデル・非公開評価コードを呼びません。単位はRun、品質分母は57 ID。欠測はnullです。

In [1]:
from pathlib import Path
import json,sqlite3,statistics,collections
base=Path.cwd()
rows=json.loads((base/'runs.json').read_text(encoding='utf-8'))
summary=json.loads((base/'summary.json').read_text(encoding='utf-8'))
started=[r for r in rows if r['run_id']]
assert len(rows)==20
assert len({r['planned_run'] for r in rows})==20
assert len({r['run_id'] for r in started})==len(started)
assert all(sum(r['condition']==c for r in rows)==10 for c in ('normal','anti'))
assert all(r['total_tokens'] is None or r['total_tokens']>=0 for r in rows)
assert all(r['quality_percent'] is None or 0<=r['quality_percent']<=100 for r in rows)
print({'planned':len(rows),'started':len(started),'total_token_missing':sum(r['total_tokens'] is None for r in started),'quality_missing':sum(r['quality_percent'] is None for r in started)})

{'planned': 20, 'started': 20, 'total_token_missing': 6, 'quality_missing': 20}


## SQLiteとの照合

Run表の行数・主キー・token/品質をJSONと照合し、ケースと評価のRun参照が孤立していないか確認します。

In [2]:
with sqlite3.connect(base/'analysis.sqlite') as db:
    counts={name:db.execute('SELECT COUNT(*) FROM '+name).fetchone()[0] for name in ('runs','case_results','telemetry_refs','evaluations','provenance')}
    stored={r[0]:r[1:] for r in db.execute('SELECT planned_run,run_id,total_tokens,quality_percent FROM runs')}
    assert len(stored)==len(rows)
    for row in rows:
        assert stored[row['planned_run']]==(row['run_id'],row['total_tokens'],row['quality_percent'])
    for table in ('case_results','evaluations','telemetry_refs'):
        assert db.execute('SELECT COUNT(*) FROM '+table+' c LEFT JOIN runs r ON c.run_id=r.run_id WHERE r.run_id IS NULL').fetchone()[0]==0
print(counts)

{'runs': 20, 'case_results': 1160, 'telemetry_refs': 20, 'evaluations': 20, 'provenance': 1}


In [3]:
with sqlite3.connect(base/'analysis.sqlite') as db:
    cases=dict(db.execute('SELECT run_id,COUNT(*) FROM case_results GROUP BY run_id'))
for row in started:
    if row['evaluation_completed']:
        assert row['denominator']==57
        assert sum(row[k] for k in ('passed','failed','blocked','errors'))==57
        assert cases[row['run_id']]==58
print({'evaluated_runs_checked':sum(bool(r['evaluation_completed']) for r in started),'required_ids':57,'required_cases':58})

{'evaluated_runs_checked': 20, 'required_ids': 57, 'required_cases': 58}


## 条件・モデル別の再集計

総tokenが確定したRunだけの記述統計です。欠測があるため全開始の平均とは異なります。raw pass率は有効品質ではありません。

In [4]:
checks=[]
for expected in summary['statistics']:
    group=[r for r in started if r['condition']==expected['condition'] and r['model_id']==expected['model']]
    total=[r['total_tokens'] for r in group if r['total_tokens'] is not None]
    observed=[r['observed_tokens'] for r in group if r['observed_tokens'] is not None]
    quality=[r['quality_percent'] for r in group if r['quality_percent'] is not None]
    raw=[100*r['passed']/r['denominator'] for r in group if r['evaluation_completed'] and r['passed'] is not None and r['denominator']]
    assert len(group)==expected['n'] and len(total)==expected['token_n'] and len(quality)==expected['quality_n'] and len(raw)==expected['raw_n']
    for name,values in [('token',total),('quality',quality)]:
        assert (statistics.mean(values) if values else None)==expected[name+'_mean']
        assert (statistics.median(values) if values else None)==expected[name+'_median']
    assert (statistics.mean(raw) if raw else None)==expected['raw_pass_mean_percent']
    assert (statistics.median(raw) if raw else None)==expected['raw_pass_median_percent']
    assert (statistics.mean(observed) if observed else None)==expected['observed_token_mean']
    checks.append({'model':expected['model'],'condition':expected['condition'],'n':len(group),'complete_tokens':len(total),'raw_evaluations':len(raw),'valid_quality':len(quality)})
print(json.dumps(checks,ensure_ascii=False,indent=2))

[
  {
    "model": "muse-spark-1.2-contributor",
    "condition": "normal",
    "n": 10,
    "complete_tokens": 5,
    "raw_evaluations": 10,
    "valid_quality": 0
  },
  {
    "model": "muse-spark-1.2-contributor",
    "condition": "anti",
    "n": 10,
    "complete_tokens": 9,
    "raw_evaluations": 10,
    "valid_quality": 0
  }
]


## 欠測と評価状態

親span構造、native call対応、monitor読戻しはgateway合計と独立した検証項目です。欠測や未裁定によって原本を削除しません。

In [5]:
profile=[]
for c in ('normal','anti'):
    group=[r for r in started if r['condition']==c]
    profile.append({'condition':c,'n':len(group),'total_missing':sum(r['total_tokens'] is None for r in group),'trace_incomplete':sum(r.get('trace_structure_complete') is False for r in group),'native_calls_unverified':sum(r.get('native_calls_verified') is False for r in group),'validity':dict(collections.Counter(r['evaluation_validity'] for r in group))})
print(json.dumps(profile,ensure_ascii=False,indent=2))

[
  {
    "condition": "normal",
    "n": 10,
    "total_missing": 5,
    "trace_incomplete": 1,
    "native_calls_unverified": 5,
    "validity": {
      "pending": 10
    }
  },
  {
    "condition": "anti",
    "n": 10,
    "total_missing": 1,
    "trace_incomplete": 4,
    "native_calls_unverified": 1,
    "validity": {
      "invalid": 1,
      "pending": 9
    }
  }
]


## 固定ペアの照合

事前に保存した予定表を使い、同じペアのnormalとantiを照合します。欠測を埋めず、両方が1.2の対応がある指標だけを比較します。

In [6]:
plan=json.loads((base/'planned-runs.json').read_text(encoding='utf-8'))['order']
pairs=json.loads((base/'paired-runs.json').read_text(encoding='utf-8'))
by_slot={r['planned_run']:r for r in rows}
assert len(pairs)==10 and {p['block'] for p in pairs}==set(range(1,11))
for pair in pairs:
    matched={p['condition']:by_slot[p['planned_run']] for p in plan if p['block']==pair['block']}
    n,a=matched['normal'],matched['anti']
    assert (pair['normal_run_id'],pair['anti_run_id'])==(n['run_id'],a['run_id'])
    eligible=all(r['run_id'] and r['model_id']=='muse-spark-1.2-contributor' for r in (n,a))
    assert pair['both_primary']==eligible
    for field in ('total_tokens','quality_percent'):
        expected=a[field]-n[field] if eligible and a[field] is not None and n[field] is not None else None
        assert pair[field+'_anti_minus_normal']==expected
    raw_eligible=eligible and all(r['evaluation_completed'] and r['passed'] is not None and r['denominator'] for r in (n,a))
    expected=100*a['passed']/a['denominator']-100*n['passed']/n['denominator'] if raw_eligible else None
    assert pair['raw_v6_percent_anti_minus_normal']==expected
for field,expected in summary['paired_primary_differences'].items():
    values=[p[field] for p in pairs if p[field] is not None]
    assert expected==dict(n=len(values),mean=statistics.mean(values) if values else None,median=statistics.median(values) if values else None)
token_pair_values=[p['total_tokens_anti_minus_normal'] for p in pairs if p['total_tokens_anti_minus_normal'] is not None]
assert summary['paired_token_directions']==dict(anti_lower=sum(v<0 for v in token_pair_values),equal=sum(v==0 for v in token_pair_values),anti_higher=sum(v>0 for v in token_pair_values),missing_pairs=10-len(token_pair_values))
print(summary['paired_primary_differences'])

{'total_tokens_anti_minus_normal': {'n': 4, 'mean': -1546420.5, 'median': -1707381.0}, 'quality_percent_anti_minus_normal': {'n': 0, 'mean': None, 'median': None}, 'raw_v6_percent_anti_minus_normal': {'n': 10, 'mean': -25.263157894736846, 'median': -21.05263157894737}}


## AP-001関連ケースのraw集計

台帳で関係が定義された7 IDを、元の公開ケース結果から再集計します。追加の品質点にはしません。

In [7]:
diagnostics=json.loads((base/'ap001-raw-diagnostics.json').read_text(encoding='utf-8'))['rows']
cases=[json.loads(line) for line in (base/'test-results.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(diagnostics)==14
for expected in diagnostics:
    run_ids={r['run_id'] for r in rows if r['condition']==expected['condition'] and r['evaluation_completed']}
    selected=[c for c in cases if c['run_id'] in run_ids and c['evaluation_id']==expected['evaluation_id']]
    counts=collections.Counter(c['status'] for c in selected)
    assert len(run_ids)==expected['evaluated_runs'] and len(selected)==expected['raw_cases']
    assert expected['counts']=={status:counts[status] for status in ('pass','fail','blocked','error')}
print({'diagnostic_groups_checked':len(diagnostics),'additional_quality_points':0})

{'diagnostic_groups_checked': 14, 'additional_quality_points': 0}
